# IngredientMassEstimation V2 — multiescala + profundidad estimada

Una sola imagen RGB como entrada. ConvNeXt Small (stem/etapa 1 congelados), etapas 3 y 4, Depth Anything V2 Small congelado, encoder de profundidad entrenable y dos capas Query2Label.

**Antes de empezar:** publicar la rama **v2-multiscale-depth** en GitHub y elegir GPU en Colab.
No se reanuda desde V1: es una arquitectura nueva. Se conserva la función de pérdida.

Las variables RUN_DEPTH_PREPARATION, RUN_TRAINING, RUN_CALIBRATION, RUN_TEST y RUN_INFERENCE empiezan en False. Habilitar cada operación de manera explícita.
Los mapas estimados no son profundidad métrica ni archivos depth de Nutrition5k.


## 1. GPU y Google Drive


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile

import torch
assert torch.cuda.is_available(), "Elegí Entorno de ejecución > Cambiar tipo > GPU."
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")


## 2. Descargar tu rama nueva

Para un repositorio privado, crear un secreto Colab llamado GITHUB_TOKEN, con permiso de lectura del repositorio y acceso habilitado para este notebook. Para un repositorio público, cambiar PRIVATE_REPO a False. No pegar credenciales en celdas.


In [ ]:
URL = "https://github.com/marcogloazzo/IngredientMassEstimation.git"
BRANCH = "v2-multiscale-depth"
PRIVATE_REPO = True
REPO = Path("/content/nutrition-code-v2")

def git_private(command):
    environment = os.environ.copy()
    try:
        if PRIVATE_REPO:
            from google.colab import userdata
            import base64
            token = userdata.get("GITHUB_TOKEN")
            authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            environment.update({
                "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {authorization}",
            })
            del token, authorization
        result = subprocess.run(command, env=environment, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Git falló: revisá URL, rama publicada y permisos del token. No se imprime la credencial.")
        return result.stdout.strip()
    finally:
        environment.clear()

if (REPO / ".git").is_dir():
    current = git_private(["git", "-C", str(REPO), "branch", "--show-current"])
    assert current == BRANCH, f"Esta carpeta tiene otra rama: {current}. Usá una carpeta nueva."
    git_private(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH])
else:
    git_private(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)])

PROJECT = next((p for p in (REPO, REPO / "IngredientMassEstimation")
                if (p / "config/query2label_100_v2.json").is_file()), None)
assert PROJECT is not None, "Falta publicar el código V2."
os.chdir(PROJECT)
print("Proyecto:", PROJECT)
print("Rama:", git_private(["git", "-C", str(REPO), "branch", "--show-current"]))


## 3. Dependencias y rutas separadas de V1

No copiar el .venv de Windows ni reemplazar PyTorch/CUDA de Colab.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install",
                "Pillow>=10,<13", "tqdm>=4.66,<5", "opencv-python-headless>=4.10,<4.12"], check=True)
import torchvision, numpy as np
assert hasattr(torch.amp, "GradScaler"), "Se necesita torch >= 2.3."
assert torch.cuda.is_available()
print("Torch:", torch.__version__, "| Torchvision:", torchvision.__version__)

DRIVE = Path("/content/drive/MyDrive/IngredientMassEstimation")
V2 = DRIVE / "v2"
CONFIG = PROJECT / "config/query2label_100_v2.json"
PROCESSED = PROJECT / "data/processed_100"
DATA_PARENT = Path("/content/nutrition5k")
DEPTH_REPO = Path("/content/depth-anything-v2")
DEPTH_CACHE = Path("/content/depth_v2_vits")
DEPTH_ARCHIVE = V2 / "depth_v2_vits.zip"
DEPTH_WEIGHTS = V2 / "weights/depth_anything_v2_vits.pth"
RUN = V2 / "runs/query2label_100_asl"
VALIDATION = V2 / "reports/query2label_asl_validation"
TEST = V2 / "reports/query2label_asl_test"
CHECKPOINT = VALIDATION / "calibrated.pt"
V2.mkdir(parents=True, exist_ok=True)

def run_stream(arguments):
    """Mostrar progreso en la celda mientras corre el subprocess."""
    command = [str(arg) for arg in arguments]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


## 4. Extraer el ZIP RGB a /content

Reutiliza MyDrive/IngredientMassEstimation/nutrition5k_rgb.zip. No hace falta extraerlo en Drive.
Acepta separadores / o \ de ZIP creados en Windows y busca la raíz real del dataset.


In [ ]:
from pathlib import PurePosixPath
import stat

def extract_zip_safe(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        checked = []
        for info in source.infolist():
            name = info.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if relative.is_absolute() or not relative.parts or ":" in relative.parts[0]:
                raise ValueError(f"Nombre inseguro en ZIP: {name}")
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Ruta insegura en ZIP: {name}")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"No se aceptan enlaces en ZIP: {name}")
            checked.append((info, target, name.endswith("/")))
        for info, target, is_directory in checked:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(info) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)

def dataset_roots():
    if not DATA_PARENT.is_dir():
        return []
    return sorted({folder.parent for folder in DATA_PARENT.rglob("metadata")
                   if folder.is_dir()
                   and (folder / "dish_metadata_cafe1.csv").is_file()
                   and (folder / "ingredients_metadata.csv").is_file()
                   and (folder.parent / "dish_ids").is_dir()
                   and (folder.parent / "imagery/realsense_overhead").is_dir()})

roots = dataset_roots()
if not roots:
    source_archive = DRIVE / "nutrition5k_rgb.zip"
    assert source_archive.is_file(), f"Falta subir: {source_archive}"
    local_archive = Path("/content/nutrition5k_rgb.zip")
    shutil.copy2(source_archive, local_archive)
    extract_zip_safe(local_archive, DATA_PARENT)
    roots = dataset_roots()
assert len(roots) == 1, f"No se encontró una raíz única del dataset: {roots}"
DATA = roots[0]
print("Dataset:", DATA)


## 5. Etiquetas y splits

Se conserva el seed y el split de V1 para esta comparación. No se extraen vectores ConvNeXt congelados.


In [ ]:
run_stream([sys.executable, "-u", "scripts/prepare_nutrition5k.py",
            "--config", CONFIG, "--dataset-root", DATA, "--output-dir", PROCESSED])
summary = json.loads((PROCESSED / "summary.json").read_text())
print("Clases:", summary["vocabulary_size"], "| Splits:", summary["split_counts"])
print("Referencia del mismo dataset: 2332 train, 423 validation, 507 test.")


## 6. Depth Anything V2 Small: código oficial y pesos congelados

Se descarga el código oficial en una revisión fija y los pesos Small (~100 MB) se guardan en Drive.
No se entrena Depth Anything. No se usan los pesos de OmniFood ni los modelos Base/Large de tu PC.


In [ ]:
import urllib.request

settings = json.loads(CONFIG.read_text())
revision = settings["depth"]["repo_revision"]
assert settings["depth"]["encoder"] == "vits", "Este notebook está configurado para Depth Anything Small."
if not (DEPTH_REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "https://github.com/DepthAnything/Depth-Anything-V2.git",
                    str(DEPTH_REPO)], check=True)
subprocess.run(["git", "-C", str(DEPTH_REPO), "checkout", "--detach", revision], check=True)

DEPTH_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not DEPTH_WEIGHTS.is_file():
    weight_url = ("https://huggingface.co/depth-anything/Depth-Anything-V2-Small/"
                  "resolve/main/depth_anything_v2_vits.pth")
    temporary = DEPTH_WEIGHTS.with_suffix(".download")
    urllib.request.urlretrieve(weight_url, temporary)
    temporary.replace(DEPTH_WEIGHTS)
print("Pesos congelados:", DEPTH_WEIGHTS)


## 7. Cache de profundidad y controles previos — no es entrenamiento

Primera sesión: cambiar RUN_DEPTH_PREPARATION a True. Se estima la profundidad de los RGB y se guarda un ZIP en Drive.
Sesiones siguientes: se restaura ese ZIP; dejar False. El script guarda un manifest incremental para continuar si se interrumpe en el mismo runtime.
Si se pierde el runtime antes del ZIP, habrá que recalcular los mapas faltantes en una sesión nueva.

Los mapas se producen antes de las augmentations; RGB y profundidad comparten el flip.
La profundidad es relativa, normalizada por imagen a [0,1]; no representa centímetros.


In [ ]:
RUN_DEPTH_PREPARATION = False

if not (DEPTH_CACHE / "manifest.json").is_file() and DEPTH_ARCHIVE.is_file():
    local_archive = Path("/content/depth_v2_vits.zip")
    shutil.copy2(DEPTH_ARCHIVE, local_archive)
    extract_zip_safe(local_archive, DEPTH_CACHE.parent)

if RUN_DEPTH_PREPARATION:
    run_stream([sys.executable, "-u", "scripts/prepare_depth.py",
                "--config", CONFIG, "--dataset-root", DATA, "--processed-dir", PROCESSED,
                "--depth-cache", DEPTH_CACHE, "--depth-anything-repo", DEPTH_REPO,
                "--depth-checkpoint", DEPTH_WEIGHTS, "--device", "cuda"])
    local_archive = Path(shutil.make_archive("/content/depth_v2_vits", "zip",
                                            root_dir=DEPTH_CACHE.parent, base_dir=DEPTH_CACHE.name))
    temporary = DEPTH_ARCHIVE.with_suffix(".uploading")
    shutil.copy2(local_archive, temporary)
    temporary.replace(DEPTH_ARCHIVE)
    print("Cache guardado:", DEPTH_ARCHIVE)
elif not (DEPTH_CACHE / "manifest.json").is_file():
    print("Primera sesión: habilitá RUN_DEPTH_PREPARATION y repetí esta celda.")
else:
    print("Cache restaurado. La siguiente celda verifica que esté completo.")


In [ ]:
assert (DEPTH_CACHE / "manifest.json").is_file(), "Prepará el cache en la celda 7."
run_stream([sys.executable, "-u", "scripts/prepare_depth.py", "--check-cache",
            "--config", CONFIG, "--dataset-root", DATA, "--processed-dir", PROCESSED,
            "--depth-cache", DEPTH_CACHE])
run_stream([sys.executable, "-u", "scripts/train_image_model.py", "--check-data",
            "--config", CONFIG, "--dataset-root", DATA, "--processed-dir", PROCESSED,
            "--depth-cache", DEPTH_CACHE])
print("Controles completos; todavía no se entrenó.")


## 8. Entrenamiento V2 en GPU

Cambiar RUN_TRAINING a True solo cuando quieras comenzar.
MODE="new" para la primera corrida; MODE="resume" para continuar desde last.pt de V2.
TOTAL_EPOCHS es el límite total (100), no 100 épocas adicionales al reanudar. Early stopping usa paciencia 8.
Batch 4 × acumulación 8 = efectivo 32. Warmup 3 épocas; desde la 4 se ajustan etapas 2–4.

best.pt se actualiza con el menor ingredient_mass_l1_per_plate de validación.
history.csv informa pérdidas train/validation y MAE/F1 de validación. Se muestra el progreso en vivo.
No mezclar V1/V2, ASL/BCE ni configuraciones dentro de una misma carpeta de corrida.


In [ ]:
RUN_TRAINING = False
MODE = "new"  # "new" o "resume"
TOTAL_EPOCHS = 100

if RUN_TRAINING:
    assert MODE in ("new", "resume")
    arguments = [sys.executable, "-u", "scripts/train_image_model.py",
                 "--config", CONFIG, "--dataset-root", DATA, "--processed-dir", PROCESSED,
                 "--depth-cache", DEPTH_CACHE, "--output-dir", RUN,
                 "--torch-cache", DRIVE / "torch_cache", "--device", "cuda",
                 "--workers", "2", "--epochs", str(TOTAL_EPOCHS)]
    if MODE == "resume":
        assert (RUN / "last.pt").is_file(), "Falta last.pt de V2."
        arguments += ["--resume", RUN / "last.pt"]
    else:
        assert not RUN.exists() or not any(RUN.iterdir()), "Hay una corrida: usar resume o una carpeta nueva."
    run_stream(arguments)
else:
    print("Entrenamiento desactivado. Para comenzar, habilitá RUN_TRAINING.")


### Ver historial de la corrida

Esta celda solo lee el CSV existente. La evaluación final de test está en el punto 9.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_path = RUN / "history.csv"
if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history.tail(10))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history.epoch, history.train_loss, label="Train")
    axes[0].plot(history.epoch, history.validation_loss, label="Validación")
    axes[0].set_title("Pérdida"); axes[0].legend()
    axes[1].plot(history.epoch, history.ingredient_mass_l1_per_plate, label="L1 ingredientes/plato")
    axes[1].plot(history.epoch, history.total_mass_mae, label="MAE masa total")
    axes[1].set_title("Validación — gramos"); axes[1].legend()
    axes[2].plot(history.epoch, history.micro_f1)
    axes[2].set_title("F1 presencia — validación")
    for ax in axes:
        ax.set_xlabel("Época"); ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()
else:
    print("Todavía no existe history.csv.")


## 9. Calibrar con validación y evaluar test

RUN_CALIBRATION calcula un umbral global de presencia con validación y guarda calibrated.pt.
RUN_TEST evalúa una sola vez el modelo elegido en test; no usar test para elegir epochs/variantes.
metrics.json y evaluation_details.json separan las métricas del historial de entrenamiento.


In [ ]:
RUN_CALIBRATION = False
RUN_TEST = False

shared = ["--dataset-root", DATA, "--processed-dir", PROCESSED,
          "--depth-cache", DEPTH_CACHE, "--device", "cuda", "--batch-size", "4", "--workers", "2"]
if RUN_CALIBRATION:
    assert (RUN / "best.pt").is_file(), "Falta entrenar V2."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py",
                "--checkpoint", RUN / "best.pt", "--split", "validation", "--calibrate",
                "--output-dir", VALIDATION, *shared])
if RUN_TEST:
    assert CHECKPOINT.is_file(), "Primero calibrá con validación."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py",
                "--checkpoint", CHECKPOINT, "--split", "test", "--output-dir", TEST, *shared])
if not RUN_CALIBRATION and not RUN_TEST:
    print("Calibración y test desactivados.")

if (TEST / "metrics.json").is_file():
    print("Resultados de TEST:", TEST / "metrics.json")
    display(json.loads((TEST / "metrics.json").read_text()))


## 10. Inferencia RGB y resultados visuales

Para una foto propia: subir IMG_9993.png a la carpeta photos existente en Drive.
Para un plato del dataset: usar PHOTO_SOURCE="validation" o "test" y el índice PLATE_INDEX.
El script estima internamente la profundidad, libera Depth Anything de GPU y luego ejecuta V2.

No pasar la masa real para evaluar end-to-end. Las fotos personales no se incorporan a train.
Las superposiciones de atención no son segmentación ni prueba de presencia.


In [ ]:
RUN_INFERENCE = False
PHOTO_SOURCE = "personal"  # "personal", "validation" o "test"
PLATE_INDEX = 0
PHOTO = DRIVE / "photos/IMG_9993.png"

if PHOTO_SOURCE in ("validation", "test"):
    records = [json.loads(line) for line in (PROCESSED / "records.jsonl").read_text().splitlines() if line.strip()]
    candidates = [record for record in records if record["split"] == PHOTO_SOURCE]
    assert 0 <= PLATE_INDEX < len(candidates)
    selected = candidates[PLATE_INDEX]
    PHOTO = Path("/content/v2_examples") / f"{selected['dish_id']}.png"
    PHOTO.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA / selected["image"], PHOTO)
    print("Plato:", selected["dish_id"], "| Split:", PHOTO_SOURCE)
    print("Masa anotada:", selected["total_mass"], "g")
elif PHOTO_SOURCE != "personal":
    raise ValueError("PHOTO_SOURCE debe ser personal, validation o test.")

PREDICTIONS = V2 / "predictions"
if RUN_INFERENCE:
    assert CHECKPOINT.is_file(), "Falta el checkpoint V2 calibrado."
    assert PHOTO.is_file(), f"Falta la foto: {PHOTO}"
    run_stream([sys.executable, "-u", "scripts/infer_image_model.py",
                "--checkpoint", CHECKPOINT, "--image", PHOTO, "--output-dir", PREDICTIONS,
                "--device", "cuda", "--save-attention", "--top-k", "10",
                "--depth-anything-repo", DEPTH_REPO, "--depth-checkpoint", DEPTH_WEIGHTS])
else:
    print("Inferencia desactivada.")


In [ ]:
from PIL import Image

result_dir = PREDICTIONS / PHOTO.stem
prediction_file = result_dir / "prediction.json"
if prediction_file.is_file():
    prediction = json.loads(prediction_file.read_text())
    ingredients = sorted(prediction["ingredients"], key=lambda item: item["grams"], reverse=True)[:10]
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    axes[0].imshow(Image.open(result_dir / "intermediates/input_crop.png"))
    axes[0].set_title("RGB usado por el modelo"); axes[0].axis("off")
    axes[1].imshow(Image.open(result_dir / "intermediates/estimated_relative_depth.png"), cmap="magma")
    axes[1].set_title("Profundidad relativa estimada"); axes[1].axis("off")
    axes[2].barh([item["name"] for item in ingredients][::-1],
                [item["grams"] for item in ingredients][::-1], color="#278b87")
    axes[2].set_xlabel("Gramos estimados"); axes[2].set_title("Top 10 ingredientes")
    plt.suptitle(f"Masa total estimada: {prediction['predicted_total_mass_grams']:.1f} g", fontsize=16)
    plt.tight_layout(); plt.show()
    display(pd.DataFrame(ingredients)[["name", "probability", "recognized", "grams", "share"]])
    display(prediction["nutrition"])
    overlays = sorted((result_dir / "intermediates").glob("*_attention.png"))[:10]
    if overlays:
        fig, axes = plt.subplots(2, 5, figsize=(18, 7))
        for ax, path in zip(axes.flat, overlays):
            ax.imshow(Image.open(path)); ax.set_title(path.stem.replace("_attention", "")); ax.axis("off")
        for ax in list(axes.flat)[len(overlays):]:
            ax.axis("off")
        plt.suptitle("Atención combinada de las etapas 3 y 4")
        plt.tight_layout(); plt.show()
else:
    print("Primero ejecutá la inferencia del punto 10.")


## Qué conservar

Git: código, configuración V2, pruebas, documentos y este notebook sin salidas.
Drive: reutilizar nutrition5k_rgb.zip; conservar v2/weights/, v2/depth_v2_vits.zip, v2/runs/, v2/reports/ y v2/predictions/.
No subir datasets, cache, pesos, .venv o resultados al repositorio.

Para reanudar: ejecutar preparación/restauración del runtime, dejar RUN_DEPTH_PREPARATION=False si existe el ZIP y usar MODE="resume".
Para inferencia V2 se necesitan el checkpoint del estimador y los pesos/código de Depth Anything.
V1 permanece separado e intacto. Ver COLAB_QUERY2LABEL_V2.md para la guía completa.
